# Install libs

In [1]:
!pip install vibdata==1.1.1 signalAI==0.0.8

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 119.0 MB/s eta 0:00:00


# Import Libs

In [2]:
import warnings
warnings.filterwarnings("ignore")

# Basic imports
import numpy as np
import numpy.typing as npt
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
import copy

# vibdata
import vibdata.raw as raw_datasets
from vibdata.deep.DeepDataset import DeepDataset, convertDataset
from vibdata.deep.signal.transforms import (
    Sequential,
    SplitSampleRate,
    FeatureExtractor,
    FilterByValue,
    Split
)
from vibdata.deep.signal.core import SignalSample

# SignalAI
from signalAI.experiments.features_1d import Features1DExperiment
from signalAI.utils.group_dataset import GroupDataset
from signalAI.utils.fold_idx_generator import (
    FoldIdxGeneratorUnbiased,
    FoldIdxGeneratorBiased,
)

class GroupMFPT(GroupDataset):
    NUM_FOLDS = 3

    FAKE_OUTER_RACE_270_LABEL = 100

    def __init__(self, dataset: DeepDataset, custom_name: str = None) -> None:
        super().__init__(dataset, custom_name, shuffle=True)

        keys = dataset.get_labels()
        values = dataset.get_labels_name()

        self.labels_name = dict(zip(keys, values))
        name_to_label = dict(zip(values, keys))

        metainfo = dataset.get_metainfo().copy()
        # Trick so that can differentiate from outer race label
        outer_race_270_mask = (metainfo.label == name_to_label["Outer Race"]) & (metainfo.load == 270)
        metainfo.loc[outer_race_270_mask, "label"] = GroupMFPT.FAKE_OUTER_RACE_270_LABEL

        labels_frequency = metainfo.label.value_counts()

        # Create a dict with the amount of samples per fold
        self.labels_bins = {
            label: {"samples_per_fold": np.ceil(total / GroupMFPT.NUM_FOLDS), "current_amount": 0}
            for label, total in labels_frequency.items()
        }

    def _get_group_divided(self, label):
        current_amount = self.labels_bins[label]["current_amount"]
        samples_per_fold = self.labels_bins[label]["samples_per_fold"]

        group = (current_amount // samples_per_fold) + 1
        self.labels_bins[label]["current_amount"] += 1
        return int(group-1)

    def _assigne_group(self, sample: SignalSample) -> int:
        label = sample["metainfo"]["label"]
        label_str = self.labels_name[label]
        load = sample["metainfo"]["load"]

        if label_str == "Outer Race" and load == 270:
            label = self.FAKE_OUTER_RACE_270_LABEL

        return self._get_group_divided(label)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Deep Learning Experiments

## Import MFPT dataset

In [4]:
raw_root_dir = "../data/raw_data/mfpt"
raw_dataset = raw_datasets.MFPT_raw(raw_root_dir, download=True)

Cached downloading...
Hash: md5:4631f552c6a0769996ee1d09e5feb209
From (original): https://drive.google.com/uc?id=1VxGlOMCEED7jy2qAoE9nKYAK8h5i6TIb
From (redirected): https://drive.google.com/uc?id=1VxGlOMCEED7jy2qAoE9nKYAK8h5i6TIb&confirm=t&uuid=e6d70591-61f0-4e31-8fab-44fd685dc290
To: ../data/raw_data/mfpt/MFPT_raw/MFPT.zip
100%|██████████| 61.6M/61.6M [00:01<00:00, 53.9MB/s]


## Time domain

In [5]:
transforms_time = Sequential(
    [
        SplitSampleRate()
    ]
)
print(transforms_time)

Sequential(transforms=[SplitSampleRate()])


In [6]:
deep_root_dir_time = "../data/deep_data/deep_learning"
deep_dataset_time = convertDataset(raw_dataset,filter=FilterByValue(on_field="sample_rate", values=48828),transforms=transforms_time, dir_path=deep_root_dir_time, batch_size=32)

Transformando


Converting MFPT: 100%|██████████| 1/1 [00:00<00:00,  2.49it/s]


## Generate Unbiased Folds Single Round

In [ ]:
# folds_singleround_deep = FoldIdxGeneratorUnbiased(deep_dataset_time, GroupMFPT , dataset_name="MFPT_deep").generate_folds()
# folds_singleround_deep

## Generate Unbiased Folds MultiRound

In [8]:
class GroupMultiRoundMFPT(GroupDataset):
    @staticmethod
    def _assigne_group(sample: SignalSample) -> int:
        sample_metainfo = sample["metainfo"]
        return sample_metainfo["label"].astype(str) + " " + sample_metainfo["load"].astype(int).astype(str)

CLASS_DEF = {23: "N", 25: "O", 24: "I"}
CONDITION_DEF = {"0":"C1","25":"C2","50":"C3","100":"C4","150":"C5","200":"C6","250":"C7","300":"C8"}
folds_multiround_deep = FoldIdxGeneratorUnbiased(deep_dataset_time,
                                    GroupMultiRoundMFPT ,
                                    dataset_name="MFPT_multi",
                                    multiround=True,
                                    class_def=CLASS_DEF,
                                    condition_def=CONDITION_DEF).generate_folds_unbiased_multiround()
folds_multiround_deep

Loading group dataset from: ../data/grouping/groups_CustomGroupMFPT_multi_multiround.npy
Per round splits:  7
Number of repeats:  5
Total combinations of folds: 49
Total combinations between folds 85900584
Time to generate combinations: 14.95 seconds


  4%|▍         | 3197813/77680043 [00:58<22:46, 54498.38it/s]

Total combs:  5
round:  0
fold:  0 -> I C1, O C2,  => 0
fold:  1 -> I C3, O C3,  => 1
fold:  2 -> I C4, O C4,  => 2
fold:  3 -> I C5, O C5,  => 3
fold:  4 -> I C6, O C6,  => 4
fold:  5 -> I C7, O C7,  => 5
fold:  6 -> I C8, O C8,  => 6

round:  1
fold:  0 -> I C1, O C8,  => 7
fold:  1 -> I C3, O C4,  => 8
fold:  2 -> I C4, O C7,  => 9
fold:  3 -> I C5, O C2,  => 10
fold:  4 -> I C6, O C5,  => 11
fold:  5 -> I C7, O C3,  => 12
fold:  6 -> I C8, O C6,  => 13

round:  2
fold:  0 -> I C1, O C6,  => 14
fold:  1 -> I C3, O C5,  => 15
fold:  2 -> I C4, O C3,  => 16
fold:  3 -> I C5, O C8,  => 17
fold:  4 -> I C6, O C2,  => 18
fold:  5 -> I C7, O C4,  => 19
fold:  6 -> I C8, O C7,  => 20

round:  3
fold:  0 -> I C1, O C5,  => 21
fold:  1 -> I C3, O C8,  => 22
fold:  2 -> I C4, O C2,  => 23
fold:  3 -> I C5, O C3,  => 24
fold:  4 -> I C6, O C7,  => 25
fold:  5 -> I C7, O C6,  => 26
fold:  6 -> I C8, O C4,  => 27

round:  4
fold:  0 -> I C1, O C4,  => 28
fold:  1 -> I C3, O C6,  => 29
fold:  2 -

[array([0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 0,
        0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6]),
 array([3, 3, 3, 5, 5, 5, 1, 1, 1, 4, 4, 4, 6, 6, 6, 2, 2, 2, 0, 0, 0, 0,
        0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6]),
 array([4, 4, 4, 2, 2, 2, 5, 5, 5, 1, 1, 1, 0, 0, 0, 6, 6, 6, 3, 3, 3, 0,
        0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6]),
 array([2, 2, 2, 3, 3, 3, 6, 6, 6, 0, 0, 0, 5, 5, 5, 4, 4, 4, 1, 1, 1, 0,
        0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6]),
 array([6, 6, 6, 4, 4, 4, 0, 0, 0, 2, 2, 2, 1, 1, 1, 3, 3, 3, 5, 5, 5, 0,
        0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6])]

## DeepLearning Experiments

### Utils

In [11]:
# vibclassifier/experiments/base.py
from abc import ABC, abstractmethod
import json
from typing import Optional, Dict, Any
from vibdata.raw.base import RawVibrationDataset
from vibdata.deep.signal.transforms import Transform

class Experiment(ABC):
    """Classe base abstrata para todos os experimentos de classificação de vibração."""

    def __init__(
        self,
        name: str,
        description: str,
        dataset: Optional[RawVibrationDataset] = None,
        data_transform: Optional[Transform] = None,
        feature_selector = None,
        model = None
    ):
        """
        Inicializa o experimento.

        Args:
            name: Nome identificador do experimento
            description: Descrição detalhada do experimento
            dataset: Conjunto de dados de vibração
            data_transform: Transformação a ser aplicada nos dados brutos
            data_division_method: Método de divisão dos dados (e.g., 'kfold', 'holdout')
            data_division_params: Parâmetros para o método de divisão
            feature_selector: Seletor de features (para experimentos com extração)
            model: Modelo de machine learning/deep learning
        """
        self.name = name
        self.description = description
        self.dataset = dataset
        self.data_transform = data_transform
        self.feature_selector = feature_selector
        self.model = model

        # Resultados serão armazenados aqui
        self.results = {}

    @abstractmethod
    def prepare_data(self):
        """Prepara os dados para o experimento."""
        pass

    @abstractmethod
    def run(self):
        """Executa o experimento completo."""
        pass

    def save_results(self, filepath: str):
        """Salva os resultados do experimento."""
        # Implementação básica - pode ser extendida
        with open(filepath, 'w') as f:
            json.dump(self.results, f)

    def load_results(self, filepath: str):
        """Carrega resultados de um experimento anterior."""
        with open(filepath, 'r') as f:
            self.results = json.load(f)

    def __str__(self):
        return f"Experiment: {self.name}\nDescription: {self.description}"

In [12]:
# vibclassifier/experiments/deep_torch.py
import os
import time
import json
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import numpy as np
from typing import List, Dict, Optional, Tuple, Union
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from signalAI.utils.metrics import calculate_metrics
from signalAI.utils.experiment_result import ExperimentResults, FoldResults
import copy

class TorchVibrationDataset(Dataset):
    """Wrapper to convert dataset samples into Torch tensors."""
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Função auxiliar KL
def kl_divergence(rho, rho_hat):
    rho_hat = torch.mean(rho_hat, dim=0)
    rho = torch.tensor([rho] * len(rho_hat), device=rho_hat.device)
    epsilon = 1e-7
    term1 = rho * torch.log((rho + epsilon) / (rho_hat + epsilon))
    term2 = (1 - rho) * torch.log((1 - rho + epsilon) / (1 - rho_hat + epsilon))
    return torch.sum(term1 + term2)

class DeepLearningExperiment(Experiment):
    def __init__(
        self,
        name: str,
        description: str,
        dataset,
        data_fold_idxs: List[int],
        model: nn.Module,
        criterion: Optional[nn.Module] = None,
        # Parâmetros adaptados para o autoencoder
        reconstruction_criterion: Optional[nn.Module] = None,
        recon_loss_weight: float = 1.0,
        sparsity_target: Optional[float] = None,
        sparsity_weight: float = 0.0,
        pretrain_epochs: int = 0, # Épocas de treinamento do autoencoder
        optimizer_class: Optional[torch.optim.Optimizer] = optim.Adam,
        batch_size: int = 32,
        lr: float = 1e-3,
        num_epochs: int = 20, # Épocas de treino do classificador
        val_split: float = 0.2,
        output_dir: str = "results_torch",
        device: str = "cuda" if torch.cuda.is_available() else "cpu",
        **kwargs
    ):
        super().__init__(name, description, dataset, model=model, **kwargs)
        self.data_fold_idxs = data_fold_idxs
        self.output_dir = Path(output_dir)
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.pretrain_epochs = pretrain_epochs
        self.val_split = val_split
        self.device = device
        self.optimizer_class = optimizer_class
        self.lr = lr
        self.criterion = criterion if criterion is not None else nn.CrossEntropyLoss()

        self.reconstruction_criterion = reconstruction_criterion
        self.recon_loss_weight = recon_loss_weight
        self.sparsity_target = sparsity_target
        self.sparsity_weight = sparsity_weight

        self.is_sae_task = self.sparsity_target is not None and self.sparsity_weight > 0.0
        # Define tipo do AutoEncoder utilizado
        self.is_autoencoder_task = reconstruction_criterion is not None or self.is_sae_task or "AE1D" in model.__class__.__name__

        if self.is_sae_task and self.reconstruction_criterion is None:
             print("Warning: SAE task detected but no reconstruction_criterion. Defaulting to MSELoss.")
             self.reconstruction_criterion = nn.MSELoss()

        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs")
            model = torch.nn.DataParallel(model)

        # Guarda encoder e decoder
        self.original_model = model.module if isinstance(model, nn.DataParallel) else model

        self.n_outer_folds = len(np.unique(data_fold_idxs))
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.prepare_data()

    def prepare_data(self):
        features, labels = [], []
        for sample in self.dataset:
            features.append(sample['signal'][0])
            labels.append(sample['metainfo']['label'])
        self.X = np.array(features)
        self.label_encoder = LabelEncoder()
        self.y = self.label_encoder.fit_transform(labels)

    def _train_one_fold(
        self, X_train, y_train, X_test, y_test, fold_idx: int
    ) -> FoldResults:

        train_dataset = TorchVibrationDataset(X_train, y_train)
        test_dataset = TorchVibrationDataset(X_test, y_test)
        val_size = int(self.val_split * len(train_dataset))
        train_size = len(train_dataset) - val_size
        train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size, shuffle=False)

        model = copy.deepcopy(self.model.to(self.device))
        model_core = model.module if isinstance(model, nn.DataParallel) else model

        # Verificação da estrutura de AE (encoder, decoder, classifier)
        has_ae_structure = hasattr(model_core, 'encoder') and hasattr(model_core, 'decoder') and hasattr(model_core, 'classifier')

        # Treinamento do AutoEncoder
        if self.is_autoencoder_task and self.pretrain_epochs > 0 and has_ae_structure:
            print(f"[Fold {fold_idx}] AutoEncoder training ({self.pretrain_epochs} epochs)...")

            # Otimizador Encoder + Decoder
            optimizer_ae = self.optimizer_class([
                {'params': model_core.encoder.parameters()},
                {'params': model_core.decoder.parameters()}
            ], lr=self.lr)

            for epoch in range(self.pretrain_epochs):
                model.train()
                running_recon_loss = 0.0

                for xb, _ in train_loader:
                    xb = xb.to(self.device)
                    input_data = xb

                    if any(isinstance(m, nn.Conv1d) for m in model.modules()) and xb.ndim == 2:
                        xb = xb.unsqueeze(1)
                    elif any(isinstance(m, nn.Conv2d) for m in model.modules()) and xb.ndim == 2:
                         side = int(np.sqrt(xb.shape[1])); xb = xb.view(xb.size(0), 1, side, side)

                    optimizer_ae.zero_grad()
                    outputs = model(xb) # (class, recon, [sparsity])

                    if isinstance(outputs, tuple):
                        # Foco na reconstrução
                        reconstruction = outputs[1]

                        loss = self.reconstruction_criterion(reconstruction, input_data)

                        # Adiciona esparsidade se for SAE
                        if self.is_sae_task and len(outputs) > 2:
                            latent_features = outputs[2]
                            loss += self.sparsity_weight * kl_divergence(self.sparsity_target, latent_features)

                        loss.backward()
                        optimizer_ae.step()
                        running_recon_loss += loss.item() * input_data.size(0)

                avg_recon_loss = running_recon_loss / len(train_loader.dataset)
                if (epoch + 1) % 5 == 0 or epoch == 0:
                    print(f"  [Pre-train] Epoch {epoch+1}/{self.pretrain_epochs} Recon Loss: {avg_recon_loss:.4f}")

        # Treino do classificador
        print(f"[Fold {fold_idx}] Classifier training ({self.num_epochs} epochs)...")

        # Define otimizador para a fase supervisionada
        if has_ae_structure and self.is_autoencoder_task:
            # Se for AE: Treina Encoder + Classifier (Decoder congelado ou ignorado pelo otimizador)
            optimizer_clf = self.optimizer_class([
                {'params': model_core.encoder.parameters()},
                {'params': model_core.classifier.parameters()}
            ], lr=self.lr)
        else:
            # Se for MLP/CNN padrão: Treina todos os parâmetros
            optimizer_clf = self.optimizer_class(model.parameters(), lr=self.lr)

        train_losses, val_losses = [], []

        for epoch in range(self.num_epochs):
            epoch_start = time.time()
            model.train()
            running_loss = 0.0

            for xb, yb in train_loader:
                xb, yb = xb.to(self.device), yb.to(self.device)

                # Ajuste de shape
                if any(isinstance(m, nn.Conv1d) for m in model.modules()) and xb.ndim == 2:
                     xb = xb.unsqueeze(1)
                elif any(isinstance(m, nn.Conv2d) for m in model.modules()) and xb.ndim == 2:
                     side = int(np.sqrt(xb.shape[1])); xb = xb.view(xb.size(0), 1, side, side)

                optimizer_clf.zero_grad()
                outputs = model(xb)

                # Cálculo da perda apenas de CLASSIFICAÇÃO
                if isinstance(outputs, tuple):
                    classification_output = outputs[0] # Pega apenas a classificação
                else:
                    classification_output = outputs # Modelo padrão

                loss = self.criterion(classification_output, yb)

                loss.backward()
                optimizer_clf.step()
                running_loss += loss.item() * xb.size(0)

            avg_train_loss = running_loss / len(train_loader.dataset)

            # Validação
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(self.device), yb.to(self.device)
                    # Ajuste de shape
                    if any(isinstance(m, nn.Conv1d) for m in model.modules()) and xb.ndim == 2:
                         xb = xb.unsqueeze(1)
                    elif any(isinstance(m, nn.Conv2d) for m in model.modules()) and xb.ndim == 2:
                         side = int(np.sqrt(xb.shape[1])); xb = xb.view(xb.size(0), 1, side, side)

                    outputs = model(xb)

                    if isinstance(outputs, tuple):
                        classification_output = outputs[0]
                    else:
                        classification_output = outputs

                    loss = self.criterion(classification_output, yb)
                    val_loss += loss.item() * xb.size(0)

            avg_val_loss = val_loss / len(val_loader.dataset)

            train_losses.append(avg_train_loss)
            val_losses.append(avg_val_loss)

            epoch_time = time.time() - epoch_start
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [Supervised] Epoch {epoch+1}/{self.num_epochs} Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Time: {epoch_time:.2f}s")

        plt.figure()
        plt.plot(train_losses, label="Train Loss (Clf)")
        plt.plot(val_losses, label="Val Loss (Clf)")
        plt.legend(); plt.title(f"Loss Curve - Fold {fold_idx}")
        plt.savefig(os.path.join(self.dir_path, f"loss_curve_fold{fold_idx}_{self.start_time}.png")); plt.close()

        torch.save(model_core.state_dict(), os.path.join(self.dir_path, f"model_fold{fold_idx}.pt"))

        # Teste
        y_true, y_pred, y_proba = [], [], []
        model.eval()
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                if any(isinstance(m, nn.Conv1d) for m in model.modules()) and xb.ndim == 2:
                     xb = xb.unsqueeze(1)
                elif any(isinstance(m, nn.Conv2d) for m in model.modules()) and xb.ndim == 2:
                     side = int(np.sqrt(xb.shape[1])); xb = xb.view(xb.size(0), 1, side, side)

                outputs = model(xb)
                if isinstance(outputs, tuple):
                    classification_output = outputs[0]
                else:
                    classification_output = outputs

                probs = torch.softmax(classification_output, dim=1)
                preds = torch.argmax(probs, dim=1)
                y_true.extend(yb.cpu().numpy()); y_pred.extend(preds.cpu().numpy()); y_proba.extend(probs.cpu().numpy())

        metrics = calculate_metrics(np.array(y_true), np.array(y_pred), np.array(y_proba))
        return FoldResults(fold_idx, np.array(y_true), np.array(y_pred), np.array(y_proba), metrics)

    def run(self) -> ExperimentResults:
        self.start_time = time.strftime("%Y%m%d_%H%M%S")
        self.dir_path = os.path.join(self.output_dir, f"results_{self.name}_{self.start_time}")
        os.makedirs(self.dir_path, exist_ok=True)

        results = ExperimentResults(
            experiment_name=self.name, description=self.description,
            model_name=self.original_model.__class__.__name__, feature_names=None,
            config={'n_outer_folds': self.n_outer_folds, 'pretrain_epochs': self.pretrain_epochs,
                    'finetune_epochs': self.num_epochs, 'batch_size': self.batch_size, 'lr': self.lr}
        )

        for outer_fold in range(self.n_outer_folds):
            print(f"\n=== Outer Fold {outer_fold+1}/{self.n_outer_folds} ===")
            train_mask = self.data_fold_idxs != outer_fold
            test_mask = self.data_fold_idxs == outer_fold

            try:
                fold_result = self._train_one_fold(self.X[train_mask], self.y[train_mask], self.X[test_mask], self.y[test_mask], outer_fold)
                results.add_fold_result(fold_result)
                print(f"  Result: Acc={fold_result.metrics['accuracy']:.4f}, F1={fold_result.metrics['f1']:.4f}")
            except Exception as e:
                print(f"Error in fold {outer_fold}: {e}")
                import traceback; traceback.print_exc()

        results.calculate_overall_metrics()
        results.save_json(os.path.join(self.dir_path, f"results.json"))
        print("\n=== Final Results ===")
        print(f"Mean Accuracy: {results.overall_metrics['accuracy']:.4f}")
        return results

### 1D MLP adaptado

In [13]:
class MLP1D(nn.Module):
    def __init__(self, input_length: int = 12000, num_classes: int = 4):
        super().__init__()

        # Camada adicional para adaptar a entrada de 12000 para 1024 progressivamente
        self.adaptation_layers = nn.Sequential(
            # 12000 -> 4096
            nn.Linear(input_length, 4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.7), # Dropout alto para evitar overfitting na entrada

            # 4096 -> 2048
            nn.Linear(4096, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            # 2048 -> 1024 (Conecta com a arquitetura original)
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4)
        )

        # Arquitetura original:
        self.fc3 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True)
        )

        self.fc4 = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True)
        )

        self.fc5 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True)
        )

        self.fc6 = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True)
        )

        self.fc7 = nn.Sequential(
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        out = torch.flatten(x, 1)

        # Passa pela adaptação primeiro
        out = self.adaptation_layers(out)

        # Segue o fluxo normal
        out = self.fc3(out)
        out = self.fc4(out)
        out = self.fc5(out)
        out = self.fc6(out)
        out = self.fc7(out)

        return out

In [14]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração MLP1D (MFPT) Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Instancia o modelo base MLP (será copiado a cada iteração)
base_model = MLP1D(input_length=input_length, num_classes=num_classes)

# Listas para armazenar métricas
multiround_results_MLP_MFPT = []
accuracies_MLP_MFPT = []
f1_scores_MLP_MFPT = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir pesos novos a cada rodada (evita data leakage)
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"mlp1d_mfpt_round_{round_idx}",
        description=f"1D MLP MFPT Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada (Array 1D)
        data_fold_idxs=current_folds,

        model=model_copy,

        # Hiperparâmetros de Treino
        batch_size=64,
        lr=3e-4,
        num_epochs=100,
        pretrain_epochs=0, # MLP é puramente supervisionada

        # Organização de saídas (Pasta específica para MFPT)
        output_dir=f"results_multiround_MLP1D_MFPT/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_MLP_MFPT.append(result)

    # Coleta métricas globais desta rodada
    accuracies_MLP_MFPT.append(result.overall_metrics['accuracy'])
    f1_scores_MLP_MFPT.append(result.overall_metrics.get('mean_f1', result.overall_metrics.get('f1')))

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL MLP1D MFPT MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_MLP_MFPT)
std_acc = np.std(accuracies_MLP_MFPT)
mean_f1 = np.mean(f1_scores_MLP_MFPT)
std_f1 = np.std(f1_scores_MLP_MFPT)

print(f"Rounds Executados: {len(multiround_results_MLP_MFPT)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração MLP1D (MFPT) Multiround ---
Input length: 48828
Num classes: 2
Total de Rounds: 5

>>> Iniciando Round 1/5 <<<

=== Outer Fold 1/7 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 0.6389, Val Loss: 0.6963, Time: 0.75s
  [Supervised] Epoch 5/100 Train Loss: 0.4324, Val Loss: 0.7185, Time: 0.08s
  [Supervised] Epoch 10/100 Train Loss: 0.2444, Val Loss: 0.8186, Time: 0.08s
  [Supervised] Epoch 15/100 Train Loss: 0.1980, Val Loss: 1.0078, Time: 0.08s
  [Supervised] Epoch 20/100 Train Loss: 0.1802, Val Loss: 1.2667, Time: 0.08s
  [Supervised] Epoch 25/100 Train Loss: 0.1227, Val Loss: 1.5544, Time: 0.08s
  [Supervised] Epoch 30/100 Train Loss: 0.0955, Val Loss: 1.7778, Time: 0.08s
  [Supervised] Epoch 35/100 Train Loss: 0.0856, Val Loss: 1.9645, Time: 0.08s
  [Supervised] Epoch 40/100 Train Loss: 0.0701, Val Loss: 2.0980, Time: 0.08s
  [Supervised] Epoch 45/100 Train Loss: 0.0686, Val Loss: 2.2018, Time: 0.08s
  [Supervised] Epoch 50/1

In [15]:
!cp -r "/content/results_multiround_MLP1D_MFPT" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D AutoEncoder

In [16]:
import torch
import torch.nn as nn

class AE1D(nn.Module):
    """
    Implementação do Autoencoder 1D Adaptado para 12k pontos.
    Arquitetura: 12000 -> 512 -> 256 -> 128 -> 64 (Latent) -> 128 -> 256 -> 512 -> 12000.
    """
    def __init__(self, input_length: int = 12000, latent_dim: int = 64, num_classes: int = 4, dropout_rate: float = 0.4):
        super(AE1D, self).__init__()

        # --- Encoder ---
        self.encoder = nn.Sequential(
            # Camada 1: Compressão Direta (12000 -> 512)
            nn.Linear(input_length, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),

            # Camada 2: 512 -> 256
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),

            # Camada 3: 256 -> 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),

            # Camada Latente: 128 -> 64
            nn.Linear(128, latent_dim)
        )

        # --- Decoder ---
        # Simétrico ao encoder
        self.decoder = nn.Sequential(
            # Latente -> 128
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),

            # 128 -> 256
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),

            # 256 -> 512
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),

            # Reconstrução Final: 512 -> 12000
            nn.Linear(512, input_length)
        )

        # Classificador (Fine-tuning)
        self.classifier = nn.Linear(latent_dim, num_classes)

    def forward(self, x):
        # Flatten para garantir (Batch, 12000)
        x = torch.flatten(x, 1)

        # Encoder
        latent_features = self.encoder(x)

        # Decoder (Reconstrução do sinal de 12k)
        reconstruction = self.decoder(latent_features)

        # Classifier
        classification_output = self.classifier(latent_features)

        return classification_output, reconstruction, latent_features

In [17]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração AE1D (MFPT) Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Critérios (Definidos uma vez, reutilizados)
classification_criterion = nn.CrossEntropyLoss()
reconstruction_criterion = nn.MSELoss()

# Instancia o modelo base AE (será copiado a cada iteração)
base_model = AE1D(input_length=input_length, latent_dim=64, num_classes=num_classes)

# Listas para armazenar métricas
multiround_results_AE_MFPT = []
accuracies_AE_MFPT = []
f1_scores_AE_MFPT = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir pesos novos a cada rodada (evita data leakage)
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"ae1d_mfpt_round_{round_idx}",
        description=f"1D AE MFPT Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada (Array 1D)
        data_fold_idxs=current_folds,

        model=model_copy,

        # Critérios de Perda
        reconstruction_criterion=reconstruction_criterion, # Critério AE
        criterion=classification_criterion,                # Critério Classificador
        recon_loss_weight=1.0,

        # Hiperparâmetros de Treino
        pretrain_epochs=100,  # Fase 1: Encoder+Decoder (Reconstrução)
        num_epochs=100,       # Fase 2: Encoder+Classifier (Classificação)
        batch_size=64,
        lr=3e-4,

        # Organização de saídas (Pasta específica para MFPT)
        output_dir=f"results_multiround_AE1D_MFPT/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_AE_MFPT.append(result)

    # Coleta métricas globais desta rodada
    accuracies_AE_MFPT.append(result.overall_metrics['accuracy'])
    f1_scores_AE_MFPT.append(result.overall_metrics.get('mean_f1', result.overall_metrics.get('f1')))

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL AE1D MFPT MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_AE_MFPT)
std_acc = np.std(accuracies_AE_MFPT)
mean_f1 = np.mean(f1_scores_AE_MFPT)
std_f1 = np.std(f1_scores_AE_MFPT)

print(f"Rounds Executados: {len(multiround_results_AE_MFPT)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração AE1D (MFPT) Multiround ---
Input length: 48828
Num classes: 2
Total de Rounds: 5

>>> Iniciando Round 1/5 <<<

=== Outer Fold 1/7 ===
[Fold 0] AutoEncoder training (100 epochs)...
  [Pre-train] Epoch 1/100 Recon Loss: 2.5467
  [Pre-train] Epoch 5/100 Recon Loss: 2.4332
  [Pre-train] Epoch 10/100 Recon Loss: 2.3538
  [Pre-train] Epoch 15/100 Recon Loss: 2.2848
  [Pre-train] Epoch 20/100 Recon Loss: 2.2294
  [Pre-train] Epoch 25/100 Recon Loss: 2.1731
  [Pre-train] Epoch 30/100 Recon Loss: 2.0998
  [Pre-train] Epoch 35/100 Recon Loss: 2.0378
  [Pre-train] Epoch 40/100 Recon Loss: 1.9634
  [Pre-train] Epoch 45/100 Recon Loss: 1.8434
  [Pre-train] Epoch 50/100 Recon Loss: 1.7470
  [Pre-train] Epoch 55/100 Recon Loss: 1.6465
  [Pre-train] Epoch 60/100 Recon Loss: 1.5565
  [Pre-train] Epoch 65/100 Recon Loss: 1.4609
  [Pre-train] Epoch 70/100 Recon Loss: 1.3549
  [Pre-train] Epoch 75/100 Recon Loss: 1.3227
  [Pre-train] Epoch 80/100 Recon Loss: 1.1975
  [Pre-train] Epoch 85/

In [18]:
!cp -r "/content/results_multiround_AE1D_MFPT" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D Sparse AutoEncoder

In [19]:
class SAE1D(nn.Module):
    """
    Implementação do Sparse Autoencoder 1D adaptado para 12k pontos.
    Arquitetura: 12000 -> 512 -> 256 -> 128 -> 64 (Latent).
    """
    def __init__(self, input_length: int = 12000, latent_dim: int = 64, num_classes: int = 4):
        super(SAE1D, self).__init__()

        # --- Encoder ---
        self.encoder = nn.Sequential(
            # Camada 1: Compressão Direta (12000 -> 512)
            # Redução drástica necessária para viabilidade
            nn.Linear(input_length, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2), # Adicionado Dropout leve

            # Camada 2: 512 -> 256
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            # Camada 3: 256 -> 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            # Camada Latente: 128 -> Latent Dim
            nn.Linear(128, latent_dim)
        )

        # A Sigmoid é OBRIGATÓRIA para SAE se você usar KL Divergence Loss
        # Ela força os neurônios latentes a ficarem entre [0, 1] (probabilidade de ativação)
        self.sparsity_activation = nn.Sigmoid()

        # --- Decoder ---
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),

            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),

            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),

            # Reconstrução: 512 -> 12000
            nn.Linear(512, input_length)
        )

        # Classificador (Fine-tuning)
        self.classifier = nn.Linear(latent_dim, num_classes)

    def forward(self, x):
        # Garante (Batch, 12000)
        x = torch.flatten(x, 1)

        # 1. Codificação Linear
        features = self.encoder(x)

        # 2. Ativação Esparsa (Latent Space)
        # Importante: A saída aqui estará entre 0 e 1
        latent_features = self.sparsity_activation(features)

        # 3. Reconstrução
        reconstruction = self.decoder(latent_features)

        # 4. Classificação
        classification_output = self.classifier(latent_features)

        return classification_output, reconstruction, latent_features

In [20]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração SAE1D (MFPT) Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Critérios (Definidos uma vez)
classification_criterion = nn.CrossEntropyLoss()
reconstruction_criterion = nn.MSELoss()

# Instancia o modelo base SAE (será copiado a cada iteração)
base_model = SAE1D(input_length=input_length, latent_dim=64, num_classes=num_classes)

# Listas para armazenar métricas
multiround_results_SAE_MFPT = []
accuracies_SAE_MFPT = []
f1_scores_SAE_MFPT = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir pesos novos a cada rodada (evita data leakage)
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"sae1d_mfpt_round_{round_idx}",
        description=f"1D Sparse AE MFPT Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada (Array 1D)
        data_fold_idxs=current_folds,

        model=model_copy,

        # Critérios de Perda
        reconstruction_criterion=reconstruction_criterion, # AE Loss
        criterion=classification_criterion,                # Clf Loss
        recon_loss_weight=1.0,

        # --- Parâmetros Específicos do SAE ---
        sparsity_target=0.05,  # (Rho) Target sparsity (5%)
        sparsity_weight=1.0,   # (Beta) Peso da penalidade KL

        # Hiperparâmetros de Treino
        pretrain_epochs=100,  # Fase 1: MSE + KL Divergence
        num_epochs=100,       # Fase 2: CrossEntropy
        batch_size=64,
        lr=3e-4,

        # Organização de saídas (Pasta específica para MFPT)
        output_dir=f"results_multiround_SAE1D_MFPT/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_SAE_MFPT.append(result)

    # Coleta métricas globais desta rodada
    accuracies_SAE_MFPT.append(result.overall_metrics['accuracy'])
    f1_scores_SAE_MFPT.append(result.overall_metrics.get('mean_f1', result.overall_metrics.get('f1')))

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL SAE1D MFPT MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_SAE_MFPT)
std_acc = np.std(accuracies_SAE_MFPT)
mean_f1 = np.mean(f1_scores_SAE_MFPT)
std_f1 = np.std(f1_scores_SAE_MFPT)

print(f"Rounds Executados: {len(multiround_results_SAE_MFPT)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração SAE1D (MFPT) Multiround ---
Input length: 48828
Num classes: 2
Total de Rounds: 5

>>> Iniciando Round 1/5 <<<

=== Outer Fold 1/7 ===
[Fold 0] AutoEncoder training (100 epochs)...
  [Pre-train] Epoch 1/100 Recon Loss: 34.6584
  [Pre-train] Epoch 5/100 Recon Loss: 32.6865
  [Pre-train] Epoch 10/100 Recon Loss: 30.4949
  [Pre-train] Epoch 15/100 Recon Loss: 28.4070
  [Pre-train] Epoch 20/100 Recon Loss: 26.4700
  [Pre-train] Epoch 25/100 Recon Loss: 24.9474
  [Pre-train] Epoch 30/100 Recon Loss: 22.9746
  [Pre-train] Epoch 35/100 Recon Loss: 21.4257
  [Pre-train] Epoch 40/100 Recon Loss: 19.5787
  [Pre-train] Epoch 45/100 Recon Loss: 18.0374
  [Pre-train] Epoch 50/100 Recon Loss: 16.4818
  [Pre-train] Epoch 55/100 Recon Loss: 15.4292
  [Pre-train] Epoch 60/100 Recon Loss: 13.8471
  [Pre-train] Epoch 65/100 Recon Loss: 13.0017
  [Pre-train] Epoch 70/100 Recon Loss: 11.7741
  [Pre-train] Epoch 75/100 Recon Loss: 10.4254
  [Pre-train] Epoch 80/100 Recon Loss: 9.5551
  [Pre

In [21]:
!cp -r "/content/results_multiround_SAE1D_MFPT" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D Denoising AutoEncoder

In [22]:
class DAE1D(nn.Module):
    """
    Implementação do Denoising Autoencoder (DAE) adaptado para 12k pontos.
    Arquitetura: 12000 -> 512 -> 256 -> 128 -> 64 (Latent).
    """
    def __init__(self, input_length: int = 12000, latent_dim: int = 64, num_classes: int = 4, noise_factor: float = 0.5):
        super(DAE1D, self).__init__()
        self.noise_factor = noise_factor

        # Encoder
        self.encoder = nn.Sequential(
            # Camada 1: Compressão Direta (12000 -> 512)
            nn.Linear(input_length, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            # Camada 2: 512 -> 256
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            # Camada 3: 256 -> 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            # Latent
            nn.Linear(128, latent_dim)
        )

        # Decoder
        # Simétrico
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),

            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),

            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),

            # Reconstrução: 512 -> 12000
            nn.Linear(512, input_length)
        )

        # --- Classificador ---
        self.classifier = nn.Linear(latent_dim, num_classes)

    def forward(self, x):
        # (Batch, 12000)
        x = torch.flatten(x, 1)

        # Denoising Injection
        if self.training:
            # Adiciona ruído apenas durante o treino
            noise = torch.randn_like(x) * self.noise_factor
            x_noisy = x + noise
        else:
            x_noisy = x

        # Encoder
        latent_features = self.encoder(x_noisy)

        # Decoder
        reconstruction = self.decoder(latent_features)

        # Classifier
        classification_output = self.classifier(latent_features)

        # (Classification, Reconstruction, Latent[Opcional])
        return classification_output, reconstruction, latent_features

In [23]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração DAE1D (MFPT) Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Critérios (Definidos uma vez)
classification_criterion = nn.CrossEntropyLoss()
reconstruction_criterion = nn.MSELoss()

# Instancia o modelo base DAE (será copiado a cada iteração)
# noise_factor=0.5 define a intensidade do ruído injetado no treino
base_model = DAE1D(input_length=input_length, latent_dim=64, num_classes=num_classes, noise_factor=0.5)

# Listas para armazenar métricas
multiround_results_DAE_MFPT = []
accuracies_DAE_MFPT = []
f1_scores_DAE_MFPT = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir pesos novos a cada rodada
    # Importante para DAE: Garante que o modelo não "lembre" como limpar o ruído da rodada anterior
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"dae1d_mfpt_round_{round_idx}",
        description=f"1D Denoising AE MFPT Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada (Array 1D)
        data_fold_idxs=current_folds,

        model=model_copy,

        # Critérios de Perda
        reconstruction_criterion=reconstruction_criterion, # Denoising Loss
        criterion=classification_criterion,                # Classifier Loss
        recon_loss_weight=1.0,

        # Hiperparâmetros de Treino
        pretrain_epochs=100,   # Fase 1: Denoising (MSE)
        num_epochs=100,        # Fase 2: Classificação (CrossEntropy)
        batch_size=64,
        lr=3e-4,

        # Organização de saídas (Pasta específica para MFPT)
        output_dir=f"results_multiround_DAE1D_MFPT/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_DAE_MFPT.append(result)

    # Coleta métricas globais desta rodada
    accuracies_DAE_MFPT.append(result.overall_metrics['accuracy'])
    f1_scores_DAE_MFPT.append(result.overall_metrics.get('mean_f1', result.overall_metrics.get('f1')))

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL DAE1D MFPT MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_DAE_MFPT)
std_acc = np.std(accuracies_DAE_MFPT)
mean_f1 = np.mean(f1_scores_DAE_MFPT)
std_f1 = np.std(f1_scores_DAE_MFPT)

print(f"Rounds Executados: {len(multiround_results_DAE_MFPT)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração DAE1D (MFPT) Multiround ---
Input length: 48828
Num classes: 2
Total de Rounds: 5

>>> Iniciando Round 1/5 <<<

=== Outer Fold 1/7 ===
[Fold 0] AutoEncoder training (100 epochs)...
  [Pre-train] Epoch 1/100 Recon Loss: 2.6923
  [Pre-train] Epoch 5/100 Recon Loss: 2.5625
  [Pre-train] Epoch 10/100 Recon Loss: 2.4475
  [Pre-train] Epoch 15/100 Recon Loss: 2.3361
  [Pre-train] Epoch 20/100 Recon Loss: 2.2337
  [Pre-train] Epoch 25/100 Recon Loss: 2.1111
  [Pre-train] Epoch 30/100 Recon Loss: 1.9892
  [Pre-train] Epoch 35/100 Recon Loss: 1.8409
  [Pre-train] Epoch 40/100 Recon Loss: 1.7043
  [Pre-train] Epoch 45/100 Recon Loss: 1.6001
  [Pre-train] Epoch 50/100 Recon Loss: 1.4484
  [Pre-train] Epoch 55/100 Recon Loss: 1.3218
  [Pre-train] Epoch 60/100 Recon Loss: 1.2090
  [Pre-train] Epoch 65/100 Recon Loss: 1.1021
  [Pre-train] Epoch 70/100 Recon Loss: 1.0274
  [Pre-train] Epoch 75/100 Recon Loss: 0.9463
  [Pre-train] Epoch 80/100 Recon Loss: 0.8581
  [Pre-train] Epoch 85

In [24]:
!cp -r "/content/results_multiround_DAE1D_MFPT" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D CNN

In [25]:
class CNN1D(nn.Module):
    def __init__(self, input_length: int, num_classes: int):
        """
        1D CNN for vibration signal classification.
        Args:
            input_length: length of the input signal
            num_classes: number of output classes
        """
        super(CNN1D, self).__init__()

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(16)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(32)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(64)
        self.pool3 = nn.AdaptiveMaxPool1d(16)  # reduce dynamically to fixed size

        # compute flattened size
        example_input = torch.zeros(1, 1, input_length)  # [B, C, L]
        with torch.no_grad():
            x = self.pool1(F.relu(self.bn1(self.conv1(example_input))))
            x = self.pool2(F.relu(self.bn2(self.conv2(x))))
            x = self.pool3(F.relu(self.bn3(self.conv3(x))))
            flattened_size = x.shape[1] * x.shape[2]

        self.fc1 = nn.Linear(flattened_size, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # x shape: [B, L] or [B, 1, L]
        if x.ndim == 2:
            x = x.unsqueeze(1)  # add channel dim

        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [26]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração CNN1D (MFPT) Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Instancia o modelo base CNN (será copiado a cada iteração)
base_model = CNN1D(input_length=input_length, num_classes=num_classes)

# Listas para armazenar métricas
multiround_results_CNN_MFPT = []
accuracies_CNN_MFPT = []
f1_scores_CNN_MFPT = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir pesos novos a cada rodada (evita data leakage)
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"cnn1d_mfpt_round_{round_idx}",
        description=f"1D CNN MFPT Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada (Array 1D)
        data_fold_idxs=current_folds,

        model=model_copy,

        # Hiperparâmetros de Treino
        batch_size=64,
        lr=3e-4,
        num_epochs=100,
        pretrain_epochs=0, # CNN é puramente supervisionada

        # Organização de saídas (Pasta específica para MFPT)
        output_dir=f"results_multiround_CNN1D_MFPT/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_CNN_MFPT.append(result)

    # Coleta métricas globais desta rodada
    accuracies_CNN_MFPT.append(result.overall_metrics['accuracy'])
    f1_scores_CNN_MFPT.append(result.overall_metrics.get('mean_f1', result.overall_metrics.get('f1')))

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL CNN1D MFPT MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_CNN_MFPT)
std_acc = np.std(accuracies_CNN_MFPT)
mean_f1 = np.mean(f1_scores_CNN_MFPT)
std_f1 = np.std(f1_scores_CNN_MFPT)

print(f"Rounds Executados: {len(multiround_results_CNN_MFPT)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração CNN1D (MFPT) Multiround ---
Input length: 48828
Num classes: 2
Total de Rounds: 5

>>> Iniciando Round 1/5 <<<

=== Outer Fold 1/7 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 2.2329, Val Loss: 0.7800, Time: 1.07s
  [Supervised] Epoch 5/100 Train Loss: 0.7541, Val Loss: 0.7761, Time: 0.04s
  [Supervised] Epoch 10/100 Train Loss: 0.4142, Val Loss: 0.6799, Time: 0.04s
  [Supervised] Epoch 15/100 Train Loss: 0.3262, Val Loss: 0.7116, Time: 0.04s
  [Supervised] Epoch 20/100 Train Loss: 0.2478, Val Loss: 0.4372, Time: 0.04s
  [Supervised] Epoch 25/100 Train Loss: 0.1262, Val Loss: 0.4048, Time: 0.04s
  [Supervised] Epoch 30/100 Train Loss: 0.0995, Val Loss: 0.3625, Time: 0.04s
  [Supervised] Epoch 35/100 Train Loss: 0.0833, Val Loss: 0.2042, Time: 0.04s
  [Supervised] Epoch 40/100 Train Loss: 0.0423, Val Loss: 0.1325, Time: 0.04s
  [Supervised] Epoch 45/100 Train Loss: 0.0375, Val Loss: 0.1227, Time: 0.04s
  [Supervised] Epoch 50/1

In [27]:
!cp -r "/content/results_multiround_CNN1D_MFPT" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D LeNet

In [28]:
class LeNet1D(nn.Module):
    def __init__(self, in_channel=1, out_channel=4):
        super(LeNet1D, self).__init__()

        # --- Bloco Convolucional 1 ---
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channel, 6, kernel_size=64, stride=4, padding=30),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2), # 3000 -> 1500
        )

        # --- Bloco Convolucional 2 ---
        self.conv2 = nn.Sequential(
            nn.Conv1d(6, 16, kernel_size=5, stride=1, padding=0),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(12)
        )

        # --- Classificador (Fully Connected) ---
        # Input achatado: 16 canais * 12 pontos = 192 features
        self.fc1 = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 12, 120), # Tamanho clássico da LeNet-5
            nn.ReLU()
        )

        self.fc2 = nn.Sequential(
            nn.Linear(120, 84), # Tamanho clássico da LeNet-5
            nn.ReLU()
        )

        self.fc3 = nn.Linear(84, out_channel)

    def forward(self, x):
        # Garante dimensão de canal (Batch, 1, 12000)
        if x.ndim == 2:
            x = x.unsqueeze(1)

        x = self.conv1(x)
        x = self.conv2(x)

        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return x

In [29]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração LeNet1D (MFPT) Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Instancia o modelo base LeNet (será copiado a cada iteração)
base_model = LeNet1D(in_channel=1, out_channel=num_classes)

# Listas para armazenar métricas
multiround_results_LeNet_MFPT = []
accuracies_LeNet_MFPT = []
f1_scores_LeNet_MFPT = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir pesos novos a cada rodada (evita data leakage)
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"lenet1d_mfpt_round_{round_idx}",
        description=f"1D LeNet MFPT Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada (Array 1D)
        data_fold_idxs=current_folds,

        model=model_copy,

        # Hiperparâmetros de Treino
        batch_size=64,
        lr=3e-4,
        num_epochs=100,
        pretrain_epochs=0, # LeNet é puramente supervisionada

        # Organização de saídas (Pasta específica para MFPT)
        output_dir=f"results_multiround_LeNet1D_MFPT/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_LeNet_MFPT.append(result)

    # Coleta métricas globais desta rodada
    accuracies_LeNet_MFPT.append(result.overall_metrics['accuracy'])
    f1_scores_LeNet_MFPT.append(result.overall_metrics.get('mean_f1', result.overall_metrics.get('f1')))

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL LENET1D MFPT MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_LeNet_MFPT)
std_acc = np.std(accuracies_LeNet_MFPT)
mean_f1 = np.mean(f1_scores_LeNet_MFPT)
std_f1 = np.std(f1_scores_LeNet_MFPT)

print(f"Rounds Executados: {len(multiround_results_LeNet_MFPT)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração LeNet1D (MFPT) Multiround ---
Input length: 48828
Num classes: 2
Total de Rounds: 5

>>> Iniciando Round 1/5 <<<

=== Outer Fold 1/7 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 0.6925, Val Loss: 0.6422, Time: 0.08s
  [Supervised] Epoch 5/100 Train Loss: 0.5835, Val Loss: 0.5296, Time: 0.01s
  [Supervised] Epoch 10/100 Train Loss: 0.5460, Val Loss: 0.4792, Time: 0.01s
  [Supervised] Epoch 15/100 Train Loss: 0.5098, Val Loss: 0.4560, Time: 0.01s
  [Supervised] Epoch 20/100 Train Loss: 0.4650, Val Loss: 0.4365, Time: 0.01s
  [Supervised] Epoch 25/100 Train Loss: 0.4104, Val Loss: 0.3992, Time: 0.01s
  [Supervised] Epoch 30/100 Train Loss: 0.3497, Val Loss: 0.3518, Time: 0.01s
  [Supervised] Epoch 35/100 Train Loss: 0.2859, Val Loss: 0.3016, Time: 0.01s
  [Supervised] Epoch 40/100 Train Loss: 0.2252, Val Loss: 0.2486, Time: 0.01s
  [Supervised] Epoch 45/100 Train Loss: 0.1715, Val Loss: 0.1950, Time: 0.01s
  [Supervised] Epoch 50

In [30]:
!cp -r "/content/results_multiround_LeNet1D_MFPT" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D ResNet18

In [31]:
def conv3x1(in_planes, out_planes, stride=1):
    """3x1 convolution with padding"""
    return nn.Conv1d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv1d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x1(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x1(planes, planes)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class ResNet18(nn.Module):
    """
    Implementação da ResNet-18 adaptada para sinais 1D de 12.000 pontos.
    """
    def __init__(self, input_length=12000, in_channel=1, num_classes=10):
        super(ResNet18, self).__init__()

        # Configuração padrão da ResNet18
        block = BasicBlock
        layers = [2, 2, 2, 2] # 2 blocos por camada = 18 layers total

        self.inplanes = 64

        # STEM (Camada de Entrada)
        # Input: (Batch, 1, 12000)
        self.conv1 = nn.Conv1d(in_channel, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        # Blocos Residuais
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        # Classificador
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        # Inicialização de Pesos
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm1d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        # Tratamento de Dimensão: Garante (Batch, 1, Length)
        if x.ndim == 2:
            x = x.unsqueeze(1)

        # Stem
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # Layers
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # Head
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

In [32]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração ResNet18 (MFPT) Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Instancia o modelo base ResNet18 (será copiado a cada iteração)
# Nota: ResNet18 é um modelo profundo. Se houver erro de memória (OOM), reduza o batch_size.
base_model = ResNet18(input_length=input_length, num_classes=num_classes)

# Listas para armazenar métricas
multiround_results_ResNet_MFPT = []
accuracies_ResNet_MFPT = []
f1_scores_ResNet_MFPT = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir pesos novos a cada rodada (evita data leakage)
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"resnet18_mfpt_round_{round_idx}",
        description=f"ResNet18 MFPT Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada (Array 1D)
        data_fold_idxs=current_folds,

        model=model_copy,

        # Hiperparâmetros de Treino
        # Se der erro de memória (CUDA Out of Memory), reduza para 32 ou 16
        batch_size=64,
        lr=3e-4,
        num_epochs=100,
        pretrain_epochs=0, # ResNet é puramente supervisionada

        # Organização de saídas (Pasta específica para MFPT)
        output_dir=f"results_multiround_ResNet18_MFPT/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_ResNet_MFPT.append(result)

    # Coleta métricas globais desta rodada
    accuracies_ResNet_MFPT.append(result.overall_metrics['accuracy'])
    f1_scores_ResNet_MFPT.append(result.overall_metrics.get('mean_f1', result.overall_metrics.get('f1')))

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL RESNET18 MFPT MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_ResNet_MFPT)
std_acc = np.std(accuracies_ResNet_MFPT)
mean_f1 = np.mean(f1_scores_ResNet_MFPT)
std_f1 = np.std(f1_scores_ResNet_MFPT)

print(f"Rounds Executados: {len(multiround_results_ResNet_MFPT)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração ResNet18 (MFPT) Multiround ---
Input length: 48828
Num classes: 2
Total de Rounds: 5

>>> Iniciando Round 1/5 <<<

=== Outer Fold 1/7 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 0.6466, Val Loss: 1.8365, Time: 0.29s
  [Supervised] Epoch 5/100 Train Loss: 0.0909, Val Loss: 1.9790, Time: 0.28s
  [Supervised] Epoch 10/100 Train Loss: 0.0183, Val Loss: 2.6752, Time: 0.28s
  [Supervised] Epoch 15/100 Train Loss: 0.0043, Val Loss: 6.6041, Time: 0.28s
  [Supervised] Epoch 20/100 Train Loss: 0.0010, Val Loss: 10.4030, Time: 0.28s
  [Supervised] Epoch 25/100 Train Loss: 0.0004, Val Loss: 10.1651, Time: 0.28s
  [Supervised] Epoch 30/100 Train Loss: 0.0002, Val Loss: 6.8786, Time: 0.28s
  [Supervised] Epoch 35/100 Train Loss: 0.0002, Val Loss: 3.0385, Time: 0.28s
  [Supervised] Epoch 40/100 Train Loss: 0.0001, Val Loss: 0.3612, Time: 0.28s
  [Supervised] Epoch 45/100 Train Loss: 0.0001, Val Loss: 0.0081, Time: 0.28s
  [Supervised] Epoch

In [33]:
!cp -r "/content/results_multiround_ResNet18_MFPT" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D AlexNet

In [34]:
class AlexNet1D(nn.Module):
    def __init__(self, input_length=12000, in_channel=1, num_classes=10):
        super(AlexNet1D, self).__init__()

        self.features = nn.Sequential(
            # Conv1: Kernel 11 e Stride 4.
            # Reduz entrada 12000 -> ~3000
            nn.Conv1d(in_channel, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2), # 3000 -> 1500

            # Conv2
            nn.Conv1d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2), # 1500 -> 750

            # Conv3
            nn.Conv1d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv4
            nn.Conv1d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv5
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2), # 750 -> 375
        )

        self.avgpool = nn.AdaptiveAvgPool1d(6)

        self.classifier = nn.Sequential(
            nn.Dropout(),
            # 256 canais * 6 dimensão temporal = 1536 features
            nn.Linear(256 * 6, 1024), # Mantido 1024 conforme seu código (o paper original usa 4096)
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(1024, 1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        # Tratamento de segurança para dimensão (Batch, 1, 12000)
        if x.ndim == 2:
            x = x.unsqueeze(1)

        x = self.features(x)
        x = self.avgpool(x)

        # Flatten robusto
        x = torch.flatten(x, 1)

        x = self.classifier(x)
        return x

In [35]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração AlexNet1D (MFPT) Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Instancia o modelo base AlexNet (será copiado a cada iteração)
base_model = AlexNet1D(input_length=input_length, in_channel=1, num_classes=num_classes)

# Listas para armazenar métricas
multiround_results_AlexNet_MFPT = []
accuracies_AlexNet_MFPT = []
f1_scores_AlexNet_MFPT = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir pesos novos a cada rodada (evita data leakage)
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"alexnet_mfpt_round_{round_idx}",
        description=f"AlexNet 1D MFPT Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada (Array 1D)
        data_fold_idxs=current_folds,

        model=model_copy,

        # Hiperparâmetros de Treino
        # AlexNet geralmente roda bem com batch 64. Se der OOM, reduza para 32.
        batch_size=64,
        lr=3e-4,
        num_epochs=100,
        pretrain_epochs=0, # AlexNet é puramente supervisionada

        # Organização de saídas (Pasta específica para MFPT)
        output_dir=f"results_multiround_AlexNet1D_MFPT/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_AlexNet_MFPT.append(result)

    # Coleta métricas globais desta rodada
    accuracies_AlexNet_MFPT.append(result.overall_metrics['accuracy'])
    f1_scores_AlexNet_MFPT.append(result.overall_metrics.get('mean_f1', result.overall_metrics.get('f1')))

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL ALEXNET MFPT MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_AlexNet_MFPT)
std_acc = np.std(accuracies_AlexNet_MFPT)
mean_f1 = np.mean(f1_scores_AlexNet_MFPT)
std_f1 = np.std(f1_scores_AlexNet_MFPT)

print(f"Rounds Executados: {len(multiround_results_AlexNet_MFPT)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração AlexNet1D (MFPT) Multiround ---
Input length: 48828
Num classes: 2
Total de Rounds: 5

>>> Iniciando Round 1/5 <<<

=== Outer Fold 1/7 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 0.6936, Val Loss: 0.6891, Time: 0.08s
  [Supervised] Epoch 5/100 Train Loss: 0.6538, Val Loss: 0.7073, Time: 0.07s
  [Supervised] Epoch 10/100 Train Loss: 0.6444, Val Loss: 0.6430, Time: 0.07s
  [Supervised] Epoch 15/100 Train Loss: 0.5853, Val Loss: 0.5331, Time: 0.07s
  [Supervised] Epoch 20/100 Train Loss: 0.4136, Val Loss: 0.3425, Time: 0.07s
  [Supervised] Epoch 25/100 Train Loss: 0.2162, Val Loss: 0.1807, Time: 0.07s
  [Supervised] Epoch 30/100 Train Loss: 0.0054, Val Loss: 0.0007, Time: 0.07s
  [Supervised] Epoch 35/100 Train Loss: 0.0000, Val Loss: 0.0000, Time: 0.07s
  [Supervised] Epoch 40/100 Train Loss: 0.0000, Val Loss: 0.0000, Time: 0.07s
  [Supervised] Epoch 45/100 Train Loss: 0.0052, Val Loss: 0.0001, Time: 0.07s
  [Supervised] Epoch 

In [36]:
!cp -r "/content/results_multiround_AlexNet1D_MFPT" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D BiLSTM

In [37]:
class BiLSTM(nn.Module):
    def __init__(self, in_channel=1, out_channel=10):
        super(BiLSTM, self).__init__()

        # Hiperparâmetros Internos
        self.hidden_dim = 64
        self.kernel_num = 16
        self.num_layers = 2

        # self.V define o comprimento da sequência que entra na LSTM
        self.V = 100

        # Camada de stem
        self.embed1 = nn.Sequential(
            # Kernel grande e Stride 4 para lidar com alta taxa de amostragem
            nn.Conv1d(in_channel, self.kernel_num, kernel_size=64, stride=4, padding=30),
            nn.BatchNorm1d(self.kernel_num),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )

        self.embed2 = nn.Sequential(
            nn.Conv1d(self.kernel_num, self.kernel_num*2, kernel_size=3, padding=1),
            nn.BatchNorm1d(self.kernel_num*2),
            nn.ReLU(inplace=True),
            # AdaptiveMaxPool força a saída a ter exatamente comprimento self.V (100) garantindo entrada constante para a LSTM
            nn.AdaptiveMaxPool1d(self.V)
        )

        # Camada Recorrente (BiLSTM)
        # Input Size: kernel_num*2 (32 features por passo de tempo)
        self.bilstm = nn.LSTM(
            input_size=self.kernel_num*2,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            bidirectional=True,
            batch_first=True,
            bias=False
        )

        # Classificador
        # O input da linear é: Comprimento da Sequência (V) * (Hidden * 2 direções)
        self.hidden2label1 = nn.Sequential(
            nn.Linear(self.V * 2 * self.hidden_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5)
        )

        self.hidden2label2 = nn.Linear(512, out_channel)

    def forward(self, x):
        # (Batch, 1, 12000)
        if x.ndim == 2:
            x = x.unsqueeze(1)

        # Extração de Features Convolucionais
        x = self.embed1(x)
        x = self.embed2(x)
        # (Batch, 32, 100) -> (Batch, Channels, Time)

        # Preparação para LSTM
        # LSTM espera (Batch, Time, Features/Channels)
        x = x.permute(0, 2, 1) # (Batch, 100, 32)

        # Processamento Recorrente
        bilstm_out, _ = self.bilstm(x)
        bilstm_out = torch.tanh(bilstm_out)

        # Classificação
        bilstm_out = bilstm_out.reshape(bilstm_out.size(0), -1)

        logit = self.hidden2label1(bilstm_out)
        logit = self.hidden2label2(logit)

        return logit

In [38]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração BiLSTM (MFPT) Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Instancia o modelo base BiLSTM (será copiado a cada iteração)
base_model = BiLSTM(in_channel=1, out_channel=num_classes)

# Listas para armazenar métricas
multiround_results_BiLSTM_MFPT = []
accuracies_BiLSTM_MFPT = []
f1_scores_BiLSTM_MFPT = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir pesos novos a cada rodada (evita data leakage)
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"bilstm_mfpt_round_{round_idx}",
        description=f"BiLSTM MFPT Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada (Array 1D)
        data_fold_idxs=current_folds,

        model=model_copy,

        # Hiperparâmetros de Treino
        # AVISO: BiLSTM consome muita memória com sequências longas.
        # Se der erro de memória, reduza para 32, 16 ou 8.
        batch_size=64,
        lr=3e-4,
        num_epochs=100,
        pretrain_epochs=0,

        # Organização de saídas (Pasta específica para MFPT)
        output_dir=f"results_multiround_BiLSTM1D_MFPT/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_BiLSTM_MFPT.append(result)

    # Coleta métricas globais desta rodada
    accuracies_BiLSTM_MFPT.append(result.overall_metrics['accuracy'])
    f1_scores_BiLSTM_MFPT.append(result.overall_metrics.get('mean_f1', result.overall_metrics.get('f1')))

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL BILSTM MFPT MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_BiLSTM_MFPT)
std_acc = np.std(accuracies_BiLSTM_MFPT)
mean_f1 = np.mean(f1_scores_BiLSTM_MFPT)
std_f1 = np.std(f1_scores_BiLSTM_MFPT)

print(f"Rounds Executados: {len(multiround_results_BiLSTM_MFPT)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração BiLSTM (MFPT) Multiround ---
Input length: 48828
Num classes: 2
Total de Rounds: 5

>>> Iniciando Round 1/5 <<<

=== Outer Fold 1/7 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 0.6967, Val Loss: 0.6676, Time: 0.32s
  [Supervised] Epoch 5/100 Train Loss: 0.3808, Val Loss: 0.4646, Time: 0.01s
  [Supervised] Epoch 10/100 Train Loss: 0.0692, Val Loss: 0.1440, Time: 0.01s
  [Supervised] Epoch 15/100 Train Loss: 0.0063, Val Loss: 0.0210, Time: 0.01s
  [Supervised] Epoch 20/100 Train Loss: 0.0006, Val Loss: 0.0007, Time: 0.02s
  [Supervised] Epoch 25/100 Train Loss: 0.0000, Val Loss: 0.0000, Time: 0.01s
  [Supervised] Epoch 30/100 Train Loss: 0.0000, Val Loss: 0.0000, Time: 0.01s
  [Supervised] Epoch 35/100 Train Loss: 0.0000, Val Loss: 0.0000, Time: 0.01s
  [Supervised] Epoch 40/100 Train Loss: 0.0000, Val Loss: 0.0000, Time: 0.01s
  [Supervised] Epoch 45/100 Train Loss: 0.0000, Val Loss: 0.0000, Time: 0.01s
  [Supervised] Epoch 50/

In [39]:
!cp -r "/content/results_multiround_BiLSTM1D_MFPT" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"